# Install Dependencies

在此 cell 我們會安裝 `roboflow` 以及 `pyyaml`（用於解析 data.yaml）。

In [1]:
!pip install roboflow pyyaml

# Configuration

請保留下列變數與初始化程式碼（已由你提供的 API key）：

In [2]:
from roboflow import Roboflow

# 請勿修改下列值（已由需求提供）
rf = Roboflow(api_key="zV5iNZcr7sD0JGCwCG22")
project = rf.workspace("selenes-workspace-eckcv").project("car-9zc6g")
version = project.version(3)
# 預設下載格式為 yolov8
dataset = None  # will be set in the Download Dataset step

loading Roboflow workspace...
loading Roboflow project...


# Download Dataset

將 dataset 固定下載到 `../datasets`，格式為 `yolov8`。本 notebook 會將資料直接放到 `ai/datasets/`，且不會建立 `datasets/project-name/` 或 `datasets/version/`。


In [3]:
from pathlib import Path
import shutil

dst_root = Path("../datasets").resolve()
dst_root.mkdir(parents=True, exist_ok=True)

# 使用 Roboflow SDK 下載（不重新訓練）
print("Starting dataset download...")
dataset = version.download("yolov8")

# Roboflow 可能回傳物件或是字串路徑，嘗試解析實際路徑
def dataset_path_from_obj(obj):
    from pathlib import Path
    if obj is None:
        return None
    if hasattr(obj, "location"):
        return Path(str(obj.location)).resolve()
    if hasattr(obj, "download_location"):
        return Path(str(obj.download_location)).resolve()
    try:
        return Path(str(obj)).resolve()
    except Exception:
        return None

src = dataset_path_from_obj(dataset)
print("Roboflow reported dataset path:", src)

if src and src.exists():
    if not (src / "data.yaml").exists():
        nested = [p for p in src.iterdir() if p.is_dir() and (p / "data.yaml").exists()]
        if nested:
            src = nested[0]
            print("Found nested dataset root:", src)

if src and src.exists() and (src / "data.yaml").exists():
    for item in src.iterdir():
        dest_item = dst_root / item.name
        if item.is_dir():
            shutil.copytree(item, dest_item, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest_item)
    print("Copied dataset to", dst_root)
else:
    raise FileNotFoundError("Could not determine YOLOv8 dataset root from Roboflow response; check SDK or download location manually.")


Starting dataset download...



Extracting Dataset Version Zip to Car-3 in yolov8:: 100%|██████████| 585/585 [00:00<00:00, 5380.00it/s]


Roboflow reported dataset path: /Users/weiningyi/Documents/smart-car-inspection/ai/notebooks/Car-3
Copied dataset to /Users/weiningyi/Documents/smart-car-inspection/ai/datasets


# Verify Dataset

確認 `ai/datasets/` 已建立固定的 YOLOv8 資料結構，並驗證 `data.yaml`、train/valid/test 的 images 與 labels 資料夾。


In [4]:
from pathlib import Path
import yaml

datasets_root = Path("../datasets").resolve()
print("Datasets root:", datasets_root)

assert datasets_root.exists(), "ai/datasets/ does not exist"

data_yaml = datasets_root / "data.yaml"
assert data_yaml.exists(), "data.yaml not found in ai/datasets/"
print("Using data.yaml:", data_yaml)

with open(data_yaml, "r") as f:
    cfg = yaml.safe_load(f)

train_images_dir = datasets_root / "train" / "images"
train_labels_dir = datasets_root / "train" / "labels"
valid_images_dir = datasets_root / "valid" / "images"
valid_labels_dir = datasets_root / "valid" / "labels"
test_images_dir = datasets_root / "test" / "images"
test_labels_dir = datasets_root / "test" / "labels"

for path in [train_images_dir, train_labels_dir, valid_images_dir, valid_labels_dir, test_images_dir, test_labels_dir]:
    assert path.exists(), f"Required path not found: {path}"

print("data.yaml exists:", data_yaml.exists())
print("train images dir exists:", train_images_dir.exists())
print("train labels dir exists:", train_labels_dir.exists())
print("valid images dir exists:", valid_images_dir.exists())
print("valid labels dir exists:", valid_labels_dir.exists())
print("test images dir exists:", test_images_dir.exists())
print("test labels dir exists:", test_labels_dir.exists())

train_count = sum(1 for _ in train_images_dir.glob("*.jpg")) + sum(1 for _ in train_images_dir.glob("*.jpeg")) + sum(1 for _ in train_images_dir.glob("*.png"))
valid_count = sum(1 for _ in valid_images_dir.glob("*.jpg")) + sum(1 for _ in valid_images_dir.glob("*.jpeg")) + sum(1 for _ in valid_images_dir.glob("*.png"))
test_count = sum(1 for _ in test_images_dir.glob("*.jpg")) + sum(1 for _ in test_images_dir.glob("*.jpeg")) + sum(1 for _ in test_images_dir.glob("*.png"))

class_names = cfg.get("names") or cfg.get("class_names") or cfg.get("labels")
print("train images:", train_count)
print("valid images:", valid_count)
print("test images:", test_count)
print("class names:", class_names)


Datasets root: /Users/weiningyi/Documents/smart-car-inspection/ai/datasets
Using data.yaml: /Users/weiningyi/Documents/smart-car-inspection/ai/datasets/data.yaml
data.yaml exists: True
train images dir exists: True
train labels dir exists: True
valid images dir exists: True
valid labels dir exists: True
test images dir exists: True
test labels dir exists: True
train images: 255
valid images: 23
test images: 12
class names: ['license plate', 'wheel']
